# Data analysis, session 1A: Pandas to a portfolio
**October 8, 9:00-10:25 (85 minutes).** Then a **10-minute break**, returning at 10:35.

Our question: *What do we own, and where is the portfolio concentrated?*
We will build one small analysis, learning Series and DataFrames together instead of repeating each operation in separate tutorials.
The numbers below are a fixed teaching snapshot, not current market data.

| Time | Activity |
|---|---|
| 9:00-9:10 | Kernel, cells, Restart & Run All |
| 9:10-9:30 | Series, labels versus positions, alignment |
| 9:30-9:55 | DataFrames, inspection, filters, assignment |
| 9:55-10:15 | Grouping, joins, portfolio exposure |
| 10:15-10:25 | Pair exercise and debrief |

**Outcomes:** select the intended rows, compute with aligned labels, aggregate by sector, and check join cardinality.
Keep the [expanded Series](reference/2_pandas_series_basics.ipynb) and [DataFrame examples](reference/3_pandas_dataframes_basics.ipynb) for reference.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
plt.style.use('seaborn-v0_8-whitegrid')

## 1. A Series has values AND labels
Run the next cell, change one price, then run it again. Restart & Run All should reproduce your result.
`dtype` describes the stored values; the index identifies what each value belongs to.

In [ ]:
prices = pd.Series({'AAPL': 178.02, 'MSFT': 374.23, 'GOOGL': 139.64,
                    'AMZN': 147.92, 'NVDA': 495.22, 'META': 478.42,
                    'TSLA': 251.05, 'BRK-B': 411.50}, name='price')
display(prices)
print('dtype:', prices.dtype, '| mean price:', prices.mean())

### Predict before running: labels versus positions
Sorting changes positions, but labels travel with their values.
`.loc` selects labels; `.iloc` selects positions. Label slices include the endpoint; positional slices exclude it.

In [ ]:
ranked = prices.sort_values(ascending=False)
print('AAPL by label:', ranked.loc['AAPL'])
print('Largest price by position:', ranked.iloc[0])
display(ranked.iloc[:3])
display(prices.loc['AAPL':'GOOGL'])

### Vectorization and alignment
A Series calculation matches labels, even when the two Series are in different orders.
Multiplying raw arrays would lose that protection. A missing label produces a missing result, so inspect joins and alignment.

In [ ]:
shares = pd.Series({'MSFT': 100, 'AAPL': 200, 'NVDA': 50, 'AMZN': 150,
                    'BRK-B': 30, 'TSLA': 80, 'META': 60, 'GOOGL': 120}, name='shares')
position_values = prices * shares
weights = position_values / position_values.sum()
assert weights.notna().all() and np.isclose(weights.sum(), 1)
display(pd.DataFrame({'price': prices, 'shares': shares, 'value': position_values, 'weight': weights}))
print('Positions with weight > 15%:')
display(weights[weights > 0.15])

## 2. A DataFrame is a table of labeled columns
The dictionary keys become columns. `.head()`, `.info()`, and `.describe()` answer different questions: examples, schema/missingness, and numerical summaries.

In [ ]:
stocks = pd.DataFrame({
    'ticker': prices.index,
    'price': prices.to_numpy(),
    'market_cap_bn': [2785, 2780, 1750, 1529, 1218, 1212, 798, 896],
    'sector': ['Tech', 'Tech', 'Tech', 'Consumer', 'Tech', 'Tech', 'Auto', 'Financial'],
    'pe_ratio': [29.2, 36.1, 26.3, 58.7, 71.2, 28.9, 76.3, 9.1],
})
display(stocks.head())
stocks.info()
display(stocks.describe())

### Select, filter, and sort
`stocks['price']` is a Series; `stocks[['ticker', 'price']]` is a DataFrame.
Use parentheses around each condition and `&`, `|`, `~` for elementwise logic.

In [ ]:
large_tech = (stocks['sector'] == 'Tech') & (stocks['market_cap_bn'] > 1500)
display(stocks.loc[large_tech, ['ticker', 'price', 'market_cap_bn']])
display(stocks.loc[stocks['ticker'].isin(['AAPL', 'MSFT']), ['ticker', 'pe_ratio']])
by_price = stocks.sort_values('price', ascending=False)
print('Label 0:', by_price.loc[0, 'ticker'], '| first position:', by_price.iloc[0]['ticker'])
indexed = stocks.set_index('ticker')
display(indexed.loc[['AAPL', 'MSFT'], ['price', 'sector']])

### Assign deliberately
Work on a copy and use one `.loc[rows, column]` assignment. Avoid chained assignment such as `df[mask]['column'] = ...`.
Aggregations default to columnwise (`axis=0`); `axis=1` works across a row.

In [ ]:
working = stocks.copy()
working['market_cap_trn'] = working['market_cap_bn'] / 1000
working['watchlist'] = False
working.loc[large_tech, 'watchlist'] = True
display(working[['ticker', 'market_cap_trn', 'watchlist']])
display(working[['price', 'market_cap_bn']].mean())
# Each row's largest of two comparable price estimates; axis=1 means across columns.
quotes = pd.DataFrame({'vendor_a': [100, 200], 'vendor_b': [101, 199]})
display(quotes.max(axis=1))

## 3. Grouping and joining answer portfolio questions
`groupby` splits rows into groups, applies a calculation, and combines the results.
A merge matches **key values**. A left join keeps left keys, but duplicate right keys can multiply rows; `validate` checks our assumption.

In [ ]:
sector_summary = stocks.groupby('sector').agg(
    num_stocks=('ticker', 'size'), market_cap_bn=('market_cap_bn', 'sum'),
    avg_pe=('pe_ratio', 'mean'))
display(sector_summary)
holdings = shares.rename_axis('ticker').reset_index()
portfolio = holdings.merge(stocks, on='ticker', how='left', validate='one_to_one', indicator=True)
assert portfolio['_merge'].eq('both').all(), 'A holding is missing its price/sector.'
portfolio = portfolio.drop(columns='_merge')
portfolio['position_value'] = portfolio['shares'] * portfolio['price']
portfolio['weight_pct'] = portfolio['position_value'] / portfolio['position_value'].sum() * 100
display(portfolio.sort_values('weight_pct', ascending=False))

In [ ]:
exposure = portfolio.groupby('sector').agg(
    position_value=('position_value', 'sum'), num_stocks=('ticker', 'size'))
exposure['weight_pct'] = exposure['position_value'] / exposure['position_value'].sum() * 100
assert np.isclose(exposure['weight_pct'].sum(), 100)
display(exposure.sort_values('weight_pct', ascending=False))
# Concatenation adds rows; a merge adds information matched by keys.
first_half, second_half = stocks.iloc[:4], stocks.iloc[4:]
assert pd.concat([first_half, second_half], ignore_index=True).equals(stocks)

## 4. Pair exercise (10 minutes, including debrief)
1. Select non-Tech positions with weight above 5%.
2. Find the sector with the largest total position value.
3. Explain what would happen if the price table had two AAPL rows. Try a duplicate on a copy and inspect the merge error.

Write your answer in the blank cell before opening the worked check below.

In [ ]:
# Your analysis here

### Worked check
Explain your mask and join assumption to a partner before running.

In [ ]:
answer = portfolio.loc[(portfolio['sector'] != 'Tech') & (portfolio['weight_pct'] > 5),
                       ['ticker', 'sector', 'weight_pct']]
display(answer)
print('Largest exposure:', exposure['position_value'].idxmax())
try:
    holdings.merge(pd.concat([stocks, stocks.iloc[[0]]]), on='ticker', validate='one_to_one')
except pd.errors.MergeError:
    print('Duplicate price key caught: repair or choose a source before merging.')

## Break: 10:25-10:35
After the break, open [Visualize and clean](3_visualize_and_clean.ipynb).
We will use plots to ask whether a dataset is trustworthy before calculating risk.